## VS Code Debugging + Notebook Tips
Open this notebook with VS Code's Jupyter support to practice stepping through code.  Each code cell begins with the `# %%` marker so the same snippets can be sent to the VS Code interactive window or saved inside a `.py` file while preserving cell boundaries.


### Setting and clearing breakpoints
1. Click the gutter (left of the line numbers) next to the statements called out below to toggle a breakpoint; press `F9` to clear/toggle.
2. Use *Run ▶ Debug Cell* (or `Ctrl+Shift+D` → *Jupyter: Debug Cell*) to pause execution inside notebooks.
3. When working in `.py` files that contain `# %%` markers, select the desired cell and choose *Debug Cell* or place the caret on the function you want to inspect and press `F5` to start the debugger.
4. The variables pane and call stack automatically populate as soon as execution hits the breakpoint; you can pin variables to watch their values while stepping.


In [ ]:
# %% Notebook Breakpoints Playground
import math
from datetime import datetime

OBSERVATIONS = [
    {"target": "TRAPPIST-1e", "flux": 42.5, "distance_pc": 12.1},
    {"target": "K2-18b", "flux": 38.2, "distance_pc": 38.0},
    {"target": "GJ 486b", "flux": 33.1, "distance_pc": 8.1},
]

def compute_absolute_magnitude(entry):
    # Set a breakpoint here to step into the helper when debugging.
    distance_modulus = 5 * (math.log10(entry["distance_pc"]) - 1)
    magnitude = entry["flux"] - distance_modulus
    return magnitude

def summarize_entries(observations):
    timeline = []
    for obs in observations:  # ← breakpoint on this loop to pause between targets
        magnitude = compute_absolute_magnitude(obs)
        timeline.append(
            f"{datetime.utcnow().isoformat(timespec='seconds')} | {obs['target']:>12} | {magnitude:6.2f} mag"
        )
    return timeline

for line in summarize_entries(OBSERVATIONS):
    print(line)
print("Tip: Hit the breakpoint, inspect `obs`, then press F5 to continue.")


### Breakpoints in `.py` modules
- Copy the template below into a file such as `orbit_report.py`.
- Use *Run ▶ Start Debugging* (`F5`) to debug the entire script.
- Clearing a breakpoint works the same way: click the red dot or press `F9` while the caret sits on that line.


In [ ]:
# %% Standalone Script Template For Debugging
from textwrap import dedent

script_source = dedent('''#!/usr/bin/env python3
from datetime import datetime

def report_velocity(target: str, semi_major_axis_au: float, period_days: float) -> str:
    # Breakpoint idea: inspect `mean_motion` and `velocity` below.
    mean_motion = 2 * 3.14159 / period_days
    velocity = mean_motion * semi_major_axis_au * 1.496e8 / 86400
    timestamp = datetime.now().isoformat(timespec="seconds")
    return f"{timestamp} | {target:<12} | {velocity:7.2f} km/s"

if __name__ == "__main__":
    planets = [
        ("TRAPPIST-1e", 0.029, 6.1),
        ("GJ 486b", 0.017, 1.5),
        ("K2-18b", 0.160, 33.0),
    ]
    for target, axis, period in planets:
        print(report_velocity(target, axis, period))
''')

print(script_source)
print("Copy the snippet above into a .py file to debug it like any other module.")


### Tiny bug hunt: shape mismatch + NaNs
The next cell reproduces a broadcasting error (calibration factors with the wrong length) *and* a NaN-producing computation (square root of negative residuals).  Use VS Code breakpoints or `pdb` to inspect `flux_matrix`, `calibration`, `noise_floor`, and `residual` when the failure happens.


In [ ]:
# %% Bug reproduction: mismatch + NaNs
import numpy as np

flux_matrix = np.array(
    [
        [10.4, 12.1, 9.9],
        [11.2, 10.8, 10.5],
    ]
)
calibration = np.array([0.8, 0.9, 1.05, 0.95])  # <-- one value too many → shape mismatch
noise_floor = np.array([0.2, 0.0, 0.3])  # <-- zero invites divide-by-zero warnings
background = np.array([10.9, 11.5, 10.4])

def buggy_pipeline(flux_matrix, calibration, noise_floor, background):
    scaled = flux_matrix * calibration  # ValueError: operands could not be broadcast together
    residual = flux_matrix[0] - background  # Negative entries → NaNs after sqrt
    snr = np.sqrt(residual / noise_floor)
    return scaled, snr


buggy_pipeline(flux_matrix, calibration, noise_floor, background)

residual = flux_matrix[0] - background

with np.errstate(invalid='ignore', divide='ignore'):
    snr_preview = np.sqrt(residual / noise_floor)
print("NaNs present after sqrt?", np.isnan(snr_preview).any())
print("Values:", snr_preview)


## try except

You can also use try except to catch errors and debug them, and also to provide fallback behavior. Here's an example:

In [ ]:
try:
    buggy_pipeline(flux_matrix, calibration, noise_floor, background)
except ValueError as exc:
    print(f"Broadcasting bug -> {exc}")
    print("Hit a breakpoint on the `scaled` line to inspect shapes and fix the array length.")

In [ ]:
# %% Bug fix with debugger insights
import numpy as np

flux_matrix = np.array(
    [
        [10.4, 12.1, 9.9],
        [11.2, 10.8, 10.5],
    ]
)
calibration = np.array([0.8, 0.9, 1.05, 0.95])
noise_floor = np.array([0.2, 0.0, 0.3])
background = np.array([10.9, 11.5, 10.4])

def fixed_pipeline(flux_matrix, calibration, noise_floor, background):
    calibration = calibration[: flux_matrix.shape[1]]  # Trim or reshape based on debugger findings.
    scaled = flux_matrix * calibration
    safe_noise = np.where(noise_floor == 0, 1e-6, noise_floor)
    residual = flux_matrix - background
    residual = np.where(residual < 0, np.nan, residual)  # Flag negative values spotted in debugger.
    snr = np.sqrt(residual / safe_noise)
    clean = np.nan_to_num(snr, nan=0.0)
    return clean.mean(axis=0)

summary = fixed_pipeline(flux_matrix, calibration, noise_floor, background)
print("Clean SNR summary:", summary)


### Setup links and quick checklist
- [Install Python 3.10+](https://www.python.org/downloads/) or use `conda`/`mamba` so VS Code can discover your interpreter.
- Install the [VS Code Python extension](https://marketplace.visualstudio.com/items?itemName=ms-python.python) **and** the [Jupyter extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter); both are required for notebook debugging.
- Open the command palette and run “Python: Configure Tests” or “Python: Select Interpreter” to ensure the debugger attaches to the correct environment.
- Enable the integrated debugger for notebooks via `Jupyter: Debugging` in the command palette, then choose *Jupyter: Debug Cell* to start a debug run.


### Using `pdb` and `%debug`-style workflows

**Note: This is showing how it's possible to use `pdb` in python code without an IDE, but isn't really applicable to normal notebook usage since using VS Code's built-in debugger makes more sense ...**

- Insert `import pdb; pdb.set_trace()` (or the one-line shortcut `breakpoint()`) where you want execution to pause.  In this notebook we guard it behind a flag so the cell runs without stopping.
- To emulate `%debug`, wrap the suspicious code in `try/except` and call `pdb.post_mortem()` inside the exception handler—this reproduces the post-mortem debugger VS Code launches when you choose *Debug Cell* after a failure.
- Once inside the debugger, step with `n` (next), inspect variables with `p some_var`, or exit with `c` (continue).


In [ ]:
# %% pdb and post-mortem patterns
import pdb
import numpy as np

def orbital_velocity(semi_major_axis_au: float, period_days: float, trigger_break: bool = False) -> float:
    mean_motion = 2 * np.pi / period_days
    velocity = mean_motion * semi_major_axis_au * 1.496e8 / 86400
    if trigger_break:
        pdb.set_trace()  # Toggle `trigger_break=True` to drop into pdb.
    return velocity

def sqrt_rate(flux: float, exposure: float, enable_post_mortem: bool = True) -> float:
    try:
        rate = flux / exposure  # Raises ZeroDivisionError when exposure == 0.
        return np.sqrt(rate)
    except Exception:
        if enable_post_mortem:
            import traceback
            traceback.print_exc()
            pdb.post_mortem()  # Launch post-mortem debugger like %debug would.
        raise

print(f"Orbital velocity (no breakpoint triggered): {orbital_velocity(0.6, 10):.2f} km/s")
try:
    sqrt_rate(12.0, 0.0)
except ZeroDivisionError:
    print("ZeroDivisionError detected. Re-run with enable_post_mortem=True to inspect the stack interactively.")
